# Trace the design and leakage-safe validation splits

Level: 2. Prerequisites: Level 1: Build one ore-processing learning example. Reading time: 32 minutes.

## Objectives

Generate a small deterministic design through OreFlow's pipeline, inspect its within-case interpolation and whole-case holdout indices, and relate these protocols to the committed full-size validation.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Configure imports and record versions. This makes the method run reproducible.

In [1]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the local pipeline from the repository-relative layout.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Data

Owner: OreFlow authors; `data/derived/contract/operating_contract.json` and `data/derived/learning.json`, MIT project licence. Generated rows are simulations over authored cases, not measured plant samples. Repository: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The learning builder samples feasible inputs inside Contract 1, runs the process engine and creates physical features and targets. Interpolation holds out seeded rows inside each case. Leave-one-case-out withholds a complete case; grouped transfer additionally withholds sibling ore cases. These are not interchangeable: the latter asks whether a model transfers to a case it never saw.

Build a small seeded design for two cases. Each row is generated by OreFlow's production builder and solver, not typed into the notebook.

In [2]:
from pipeline.cases.catalog import CASES
from pipeline.methods.learning import build_design, interpolation_split, leave_one_case_out

# Generate a teaching-sized engine dataset within the declared operating contract.
contract = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))
selected = tuple(item for item in CASES if item.id in ('copper_porphyry_soft', 'iron_magnetite_fine'))
# Suppress pipeline progress timings so captured results are reproducible across runs.
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    design = build_design(contract, cases=selected, per_case=8, seed=20261026)
x, y, case_of = design['x'], design['y'], design['case']
print('design matrix:', x.shape, '| target matrix:', y.shape)
print('rows by case:', {str(name): int((case_of == name).sum()) for name in dict.fromkeys(case_of)})

design matrix: (16, 22) | target matrix: (16, 3)
rows by case: {'copper_porphyry_soft': 8, 'iron_magnetite_fine': 8}


Create both validation protocols and check their leakage boundaries. The test cases in a whole-case fold cannot appear in its training indices.

In [3]:
# Create both validation protocols and check their leakage boundaries.
train_idx, test_idx = interpolation_split(case_of, 0.25, seed=20261026)
folds = leave_one_case_out(case_of)
print('interpolation train/test:', len(train_idx), len(test_idx))
print('whole-case folds:', [(name, len(train), len(test)) for name, train, test in folds])
assert set(case_of[train_idx]) == set(case_of[test_idx])
assert all(set(case_of[train]).isdisjoint(set(case_of[test])) for _, train, test in folds)

interpolation train/test: 12 4
whole-case folds: [('copper_porphyry_soft', 8, 8), ('iron_magnetite_fine', 8, 8)]


Read the production evaluation separately from this teaching sample. The committed record was baked on all twelve cases with its full declared model settings.

In [4]:
# Read the production evaluation separately from this teaching sample.
record = json.loads((repo / 'data/derived/learning.json').read_text(encoding='utf-8'))
print('engine version:', record['engine_version'])
print('production design rows:', record['design']['rows'])
print('interpolation train/test:', record['interpolation']['train_rows'], record['interpolation']['test_rows'])
print('case holdouts:', len(record['leave_one_case_out']))
print('ore-group holdouts:', len(record['leave_one_group_out']))

engine version: 0.10.004
production design rows: 3072
interpolation train/test: 2460 612
case holdouts: 12
ore-group holdouts: 2


## Results

The small design makes the split mechanics inspectable; the full production record supplies fitted-model scores. Interpolation estimates behavior inside represented cases, while whole-case or ore-group holdout tests transfer. Neither is plant validation when all labels come from the engine.

## Summary

You ran the real design and split functions and checked leakage boundaries. Keep the illustrative sample and the full-bake evaluation distinct when interpreting results.

## References

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1214/aos/1013203451
- https://doi.org/10.1023/A:1010933404324